# Memory Hierarchy

Develop an experiment that exposes the latency and bandwidth effects of the CPU memory hierarchy.

## Objectives

Relate working-set size and access pattern to cache and main-memory behavior.

## Background

Caches exploit temporal and spatial locality; access patterns that exceed or bypass them expose slower memory levels.

## Prediction

A dependent randomized pointer-chasing workload should show increasing access latency as its working set exceeds successive cache capacities. The first transition is expected around the private L1 data-cache capacity, another around the private L2 capacity, and a further transition near the shared L3 capacity. Because CPUs 5 and 15 belong to L3 domains of 8 MiB and 16 MiB respectively, working sets between those capacities may exhibit lower latency on CPU 15. Working sets substantially larger than both L3 caches should approach main-memory latency on both CPUs.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

In [ ]:
from pathlib import Path

import pandas as pd


def read_optional(path: Path) -> str | None:
    try:
        return path.read_text().strip()
    except (FileNotFoundError, PermissionError, OSError):
        return None


def cache_entries(cpu_id: int) -> list[dict[str, object]]:
    cache_root = Path(f"/sys/devices/system/cpu/cpu{cpu_id}/cache")
    rows: list[dict[str, object]] = []

    for index_path in sorted(cache_root.glob("index*")):
        rows.append(
            {
                "cpu": cpu_id,
                "index": index_path.name,
                "level": read_optional(index_path / "level"),
                "type": read_optional(index_path / "type"),
                "size": read_optional(index_path / "size"),
                "line_size_bytes": read_optional(index_path / "coherency_line_size"),
                "ways": read_optional(index_path / "ways_of_associativity"),
                "sets": read_optional(index_path / "number_of_sets"),
                "shared_cpu_list": read_optional(index_path / "shared_cpu_list"),
            }
        )

    return rows


REPRESENTATIVE_CPUS = [5, 15]

cache_topology = pd.DataFrame(
    row for cpu_id in REPRESENTATIVE_CPUS for row in cache_entries(cpu_id)
)

for column in [
    "level",
    "line_size_bytes",
    "ways",
    "sets",
]:
    cache_topology[column] = pd.to_numeric(
        cache_topology[column],
        errors="coerce",
    ).astype("Int64")

cache_topology

,cpu,index,level,type,size,line_size_bytes,ways,sets,shared_cpu_list
0,5,index0,1,Data,64K,64,4,256,5
1,5,index1,1,Instruction,64K,64,4,256,5
2,5,index2,2,Unified,2048K,64,8,4096,5
3,5,index3,3,Unified,8192K,64,16,8192,0-9
4,15,index0,1,Data,64K,64,4,256,15
5,15,index1,1,Instruction,64K,64,4,256,15
6,15,index2,2,Unified,2048K,64,8,4096,15
7,15,index3,3,Unified,16384K,64,16,16384,10-19


In [ ]:
from pathlib import Path


def cpu_metadata(cpu_id: int) -> dict[str, object]:
    root = Path(f"/sys/devices/system/cpu/cpu{cpu_id}")

    return {
        "cpu": cpu_id,
        "cluster_id": read_optional(root / "topology/cluster_id"),
        "max_frequency_khz": read_optional(root / "cpufreq/cpuinfo_max_freq"),
        "current_frequency_khz": read_optional(root / "cpufreq/scaling_cur_freq"),
    }


representative_cpu_metadata = pd.DataFrame(
    cpu_metadata(cpu_id) for cpu_id in REPRESENTATIVE_CPUS
)

representative_cpu_metadata

,cpu,cluster_id,max_frequency_khz,current_frequency_khz
0,5,56,3900000,3900000
1,15,1144,3900000,3900000


## Observations

TODO: Record measured timing distributions and working-set sizes.

## Explanation

TODO: Relate measured transitions to cache capacity, locality, and memory bandwidth.

## Connection to LLMs

Memory hierarchy behavior influences embedding lookup, preprocessing, CPU kernels, and movement of model data.

## Further Exploration

TODO: Vary stride, working-set size, and thread placement independently.